In [1]:
# En terminal de VSCode
!pip install ultralytics wandb torch torchvision


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\jhonn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import shutil
import time
import yaml
import glob
from pathlib import Path
import torch
from ultralytics import YOLO
import wandb

In [3]:
class FineTuningDentalWandB:
    def __init__(self, base_path, modelo_2k_path):
        """
        Fine-tuning desde modelo de 2K hacia dataset completo de 9K
        
        Args:
            base_path: Ruta al dataset completo
            modelo_2k_path: Ruta al modelo entrenado con 2K imágenes
        """
        self.base_path = Path(base_path)
        self.modelo_2k_path = Path(modelo_2k_path)
        
        if not self.modelo_2k_path.exists():
            raise FileNotFoundError(f"No se encontró el modelo 2K en: {modelo_2k_path}")
        
        self.setup_paths()
        self.check_system()
        
        self.KEEP_MAP = {0: 0, 11: 1, 13: 2}
        self.class_names = {0: 'Caries', 1: 'impacted tooth', 2: 'Bone Loss'}
        
        self.best_precision = 0.0
        self.training_history = []
        
    def setup_paths(self):
        """Configurar rutas del dataset"""
        self.train_images = self.base_path / "train" / "images"
        self.train_labels_orig = self.base_path / "train" / "labels"
        self.valid_images = self.base_path / "valid" / "images"
        self.valid_labels_orig = self.base_path / "valid" / "labels"
        
        self.train_labels_3cls = self.base_path / "train" / "labels_3cls"
        self.valid_labels_3cls = self.base_path / "valid" / "labels_3cls"
        
        self.results_dir = self.base_path / "finetuning_results_9k"
        
        self.train_labels_3cls.mkdir(exist_ok=True)
        self.valid_labels_3cls.mkdir(exist_ok=True)
        self.results_dir.mkdir(exist_ok=True)
        
        print(f"Dataset base: {self.base_path}")
        print(f"Modelo 2K: {self.modelo_2k_path}")
        
    def check_system(self):
        """Verificar sistema"""
        print(f"\n--- CONFIGURACIÓN ---")
        print(f"PyTorch: {torch.__version__}")
        
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
            self.device = 'cuda'
            self.batch_size = 16
            self.workers = 4
            self.imgsz = 640
            print("Tiempo estimado: 4-6 horas (fine-tuning)")
        else:
            print("Usando CPU")
            self.device = 'cpu'
            self.batch_size = 4  # Reducido para CPU
            self.workers = 2
            self.imgsz = 416
            print("Tiempo estimado: 25-35 horas (fine-tuning en CPU)")
            print("RECOMENDACIÓN: Considera usar Google Colab con GPU")
    
    def filter_labels_to_3_classes(self):
        """Filtrar labels a 3 clases"""
        print(f"\n--- FILTRANDO LABELS ---")
        
        splits = [
            ("train", self.train_labels_orig, self.train_labels_3cls),
            ("valid", self.valid_labels_orig, self.valid_labels_3cls),
        ]
        
        total_kept = 0
        
        for split_name, in_dir, out_dir in splits:
            files_kept = 0
            
            for label_file in glob.glob(str(in_dir / "*.txt")):
                new_lines = []
                
                with open(label_file, "r") as f:
                    for line in f:
                        tokens = line.strip().split()
                        if not tokens:
                            continue
                        
                        old_class = int(float(tokens[0]))
                        if old_class in self.KEEP_MAP:
                            tokens[0] = str(self.KEEP_MAP[old_class])
                            new_lines.append(" ".join(tokens))
                
                if new_lines:
                    out_file = out_dir / Path(label_file).name
                    with open(out_file, "w") as f:
                        f.write("\n".join(new_lines) + "\n")
                    files_kept += 1
            
            print(f"  {split_name}: {files_kept} archivos")
            total_kept += files_kept
        
        return total_kept > 0
    
    def create_yaml(self):
        """Crear YAML para entrenamiento"""
        yaml_content = {
            'train': str(self.train_images).replace('\\', '/'),
            'val': str(self.valid_images).replace('\\', '/'),
            'nc': 3,
            'names': self.class_names
        }
        
        yaml_path = self.base_path / "dataset_full_finetuning.yaml"
        with open(yaml_path, 'w') as f:
            yaml.dump(yaml_content, f)
        
        print(f"YAML creado: {yaml_path}")
        return yaml_path
    
    def backup_and_replace_labels(self):
        """Backup y reemplazo de labels"""
        splits = [
            (self.train_labels_orig, self.train_labels_3cls),
            (self.valid_labels_orig, self.valid_labels_3cls),
        ]
        
        for orig_dir, filtered_dir in splits:
            backup_dir = orig_dir.parent / f"{orig_dir.name}_backup"
            
            if not backup_dir.exists():
                shutil.move(str(orig_dir), str(backup_dir))
            
            if not orig_dir.exists():
                shutil.copytree(str(filtered_dir), str(orig_dir))
    
    def restore_original_labels(self):
        """Restaurar labels originales"""
        splits = [self.train_labels_orig, self.valid_labels_orig]
        
        for orig_dir in splits:
            backup_dir = orig_dir.parent / f"{orig_dir.name}_backup"
            
            if backup_dir.exists():
                if orig_dir.exists():
                    shutil.rmtree(str(orig_dir))
                shutil.move(str(backup_dir), str(orig_dir))
    
    def initialize_wandb(self):
        """Inicializar W&B"""
        train_count = len(list(self.train_labels_3cls.glob("*.txt")))
        valid_count = len(list(self.valid_labels_3cls.glob("*.txt")))
        
        wandb.init(
            project="dental-finetuning-9k",
            name=f"finetune_{time.strftime('%Y%m%d_%H%M%S')}",
            config={
                "strategy": "fine-tuning",
                "base_model": "modelo_2k (46.1%)",
                "dataset_size": train_count,
                "epochs": 100,
                "batch_size": self.batch_size,
                "device": self.device,
                "lr0": 0.005,  # Reducido para fine-tuning
                "freeze_layers": 10
            }
        )
        
        print(f"\nW&B iniciado: {wandb.run.name}")
        print(f"Dashboard: {wandb.run.url}")
    
    def setup_callbacks(self, model):
        """Configurar callbacks con W&B"""
        current_epoch = [0]
        
        def on_train_epoch_start(trainer):
            current_epoch[0] = trainer.epoch + 1
            print(f"\n{'='*70}")
            print(f"ÉPOCA {current_epoch[0]}/100")
        
        def on_train_epoch_end(trainer):
            try:
                if hasattr(trainer, 'loss_items') and trainer.loss_items is not None:
                    losses = trainer.loss_items.detach().cpu().numpy()
                    wandb.log({
                        'train/box_loss': float(losses[0]),
                        'train/cls_loss': float(losses[1]),
                        'train/dfl_loss': float(losses[2])
                    }, step=current_epoch[0])
            except:
                pass
        
        def on_val_end(validator):
            try:
                if hasattr(validator, 'metrics') and validator.metrics:
                    if hasattr(validator.metrics, 'box'):
                        bm = validator.metrics.box
                        
                        precision = float(bm.mp) if hasattr(bm, 'mp') and bm.mp is not None else 0.0
                        recall = float(bm.mr) if hasattr(bm, 'mr') and bm.mr is not None else 0.0
                        map50 = float(bm.map50) if hasattr(bm, 'map50') and bm.map50 is not None else 0.0
                        map75 = float(bm.map) if hasattr(bm, 'map') and bm.map is not None else 0.0
                        
                        wandb.log({
                            'val/precision': precision,
                            'val/recall': recall,
                            'val/mAP50': map50,
                            'val/mAP50-95': map75
                        }, step=current_epoch[0])
                        
                        print(f"MÉTRICAS:")
                        print(f"  PRECISIÓN: {precision:.1%}")
                        print(f"  RECALL: {recall:.1%}")
                        print(f"  mAP50: {map50:.1%}")
                        
                        if precision > self.best_precision:
                            self.best_precision = precision
                            wandb.run.summary['best_precision'] = precision
                            wandb.run.summary['best_epoch'] = current_epoch[0]
                            print(f"  ⭐ NUEVO MEJOR MODELO")
                        
                        # Comparación con modelo base
                        improvement = (precision - 0.461) * 100  # 46.1% base
                        if improvement > 0:
                            print(f"  📊 Mejora vs modelo 2K: +{improvement:.1f} puntos")
            except Exception as e:
                print(f"Error en métricas: {e}")
        
        model.add_callback('on_train_epoch_start', on_train_epoch_start)
        model.add_callback('on_train_epoch_end', on_train_epoch_end)
        model.add_callback('on_val_end', on_val_end)
    
    def fine_tune(self, epochs=100):
        """Ejecutar fine-tuning"""
        print(f"\n{'='*70}")
        print(f"FINE-TUNING DESDE MODELO 2K")
        print(f"{'='*70}")
        print(f"Modelo base: 46.1% precisión (2K imágenes, 50 épocas)")
        print(f"Dataset completo: 9K+ imágenes")
        print(f"Objetivo: 75%+ precisión")
        print(f"Estrategia: Transfer learning + ajuste fino")
        
        # Filtrar labels
        if not self.filter_labels_to_3_classes():
            raise Exception("No se encontraron labels válidos")
        
        # Crear YAML
        yaml_path = self.create_yaml()
        
        # Backup labels
        self.backup_and_replace_labels()
        
        try:
            # Inicializar W&B
            self.initialize_wandb()
            
            # Cargar modelo pre-entrenado de 2K
            print(f"\nCargando modelo 2K desde: {self.modelo_2k_path}")
            model = YOLO(str(self.modelo_2k_path))
            
            # Configurar callbacks
            self.setup_callbacks(model)
            
            start_time = time.time()
            
            # Fine-tuning
            results = model.train(
                data=str(yaml_path),
                epochs=epochs,
                imgsz=self.imgsz,
                batch=self.batch_size,
                workers=self.workers,
                device=self.device,
                
                project=str(self.results_dir),
                name='finetuned_9k',
                save=True,
                patience=30,
                verbose=True,
                val=True,
                save_period=10,
                
                # PARÁMETROS CRÍTICOS DE FINE-TUNING
                resume=False,
                lr0=0.005,      # LR más bajo que entrenamiento desde cero
                lrf=0.001,      # LR final muy bajo
                warmup_epochs=5,  # Más warmup para transición suave
                freeze=10,      # Congelar primeras 10 capas (backbone pre-entrenado)
                
                # Optimización
                cache=True,
                optimizer='AdamW',
                cos_lr=True,
                
                # Data augmentation moderado
                hsv_h=0.015,
                hsv_s=0.7,
                hsv_v=0.4,
                translate=0.1,
                scale=0.5,
                fliplr=0.5,
                mosaic=1.0
            )
            
            training_time = time.time() - start_time
            
            # Resumen
            print(f"\n{'='*70}")
            print(f"FINE-TUNING COMPLETADO")
            print(f"Tiempo: {training_time/3600:.2f} horas")
            print(f"Mejor precisión: {self.best_precision:.1%}")
            print(f"Mejora vs 2K: +{(self.best_precision - 0.461)*100:.1f} puntos")
            print(f"{'='*70}")
            
            wandb.run.summary['training_time_hours'] = training_time/3600
            wandb.run.summary['improvement_vs_2k'] = (self.best_precision - 0.461) * 100
            
            # Guardar modelo
            model_path = self.save_model()
            
            wandb.finish()
            
            return model, model_path
            
        finally:
            self.restore_original_labels()
    
    def save_model(self):
        """Guardar modelo final"""
        try:
            best_model = self.results_dir / "finetuned_9k" / "weights" / "best.pt"
            
            if best_model.exists():
                final_path = self.base_path / f"modelo_finetuned_9k_{self.best_precision:.1%}.pt"
                shutil.copy2(best_model, final_path)
                
                wandb.save(str(best_model))
                
                artifact = wandb.Artifact(
                    name="dental_finetuned_9k",
                    type="model",
                    description=f"Fine-tuned desde 2K: {self.best_precision:.1%}"
                )
                artifact.add_file(str(best_model))
                wandb.log_artifact(artifact)
                
                print(f"Modelo guardado: {final_path}")
                return final_path
            
            return None
        except Exception as e:
            print(f"Error guardando: {e}")
            return None

def main_finetuning():
    """Función principal"""
    
    # RUTAS - AJUSTAR SEGÚN TU SISTEMA
    BASE_PATH = r"C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO"
    MODELO_2K = r"C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\modelo_dental_2k_final_46.1%.pt"
    
    print("FINE-TUNING: 2K → 9K DATASET")
    print("="*70)
    
    # Verificar W&B login
    try:
        import wandb
        if not wandb.api.api_key:
            print("Haciendo login en W&B...")
            wandb.login(key='ed86b89c0af5efbce45b9b3ce2b0cc318c73059b')
    except:
        pass
    
    # Crear trainer
    trainer = FineTuningDentalWandB(
        base_path=BASE_PATH,
        modelo_2k_path=MODELO_2K
    )
    
    # Fine-tuning
    model, model_path = trainer.fine_tune(epochs=100)
    
    if model:
        print(f"\nÉXITO")
        print(f"Modelo: {model_path}")
        print(f"Revisa W&B dashboard para análisis completo")
    else:
        print(f"\nFalló el entrenamiento")
    
    return model, trainer

In [4]:
if __name__ == "__main__":
    model, trainer = main_finetuning()

FINE-TUNING: 2K → 9K DATASET
Dataset base: C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO
Modelo 2K: C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\modelo_dental_2k_final_46.1%.pt

--- CONFIGURACIÓN ---
PyTorch: 2.8.0+cpu
Usando CPU
Tiempo estimado: 25-35 horas (fine-tuning en CPU)
RECOMENDACIÓN: Considera usar Google Colab con GPU

FINE-TUNING DESDE MODELO 2K
Modelo base: 46.1% precisión (2K imágenes, 50 épocas)
Dataset completo: 9K+ imágenes
Objetivo: 75%+ precisión
Estrategia: Transfer learning + ajuste fino

--- FILTRANDO LABELS ---
  train: 8881 archivos
  valid: 2725 archivos
YAML creado: C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\dataset_full_finetuning.yaml


wandb: Currently logged in as: jhonnypt64 (jhonnypt64-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



W&B iniciado: finetune_20251001_211357
Dashboard: https://wandb.ai/jhonnypt64-antenor-orrego-private-university/dental-finetuning-9k/runs/og8k0twt

Cargando modelo 2K desde: C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\modelo_dental_2k_final_46.1%.pt
New https://pypi.org/project/ultralytics/8.3.204 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.11.9 torch-2.8.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\dataset_full_finetuning.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



FINE-TUNING COMPLETADO
Tiempo: 60.33 horas
Mejor precisión: 79.0%
Mejora vs 2K: +32.9 puntos
Error guardando: [WinError 1314] El cliente no dispone de un privilegio requerido: 'C:\\Users\\jhonn\\OneDrive\\Desktop\\dataset_dientes\\el_candidato\\YOLO\\YOLO\\finetuning_results_9k\\finetuned_9k\\weights\\best.pt' -> 'c:\\Users\\jhonn\\OneDrive\\Desktop\\documentos de sedalib\\wandb\\run-20251001_211358-og8k0twt\\files\\weights\\best.pt'


train/box_loss,▅█▆▅▁▄▆▄▅▅▄▅▅▄▄▅▄▅▄▅▅▄▅▃▅▄▂▄▄▄▅▄▆▃▄▄▄▇▂▄
train/cls_loss,▄▄▂▃█▄▃▃▃█▃▄▂▃▃▃▃▄▂▄▅▃▂▂▃▃▃▂▁▃▂▃▂▄▃▁▂▆▆▁
train/dfl_loss,▅▅▅▇▅▆▆▅▅▅▄▅█▆▆▆▅▅▇▅▅▄▆▆▄▄▄▃▄▅▅▄▅▅▄▄▅▅▁▄
val/mAP50,▁▂▂▂▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇█▇████████████
val/mAP50-95,▁▄▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇██████████████████
val/precision,██▃▆█▁▁▁▁▂▂▂▂▂▂▂▃▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
val/recall,▁▂▂▄▄▇▇▇▅▇▇▇▇▇▇▇▇██▇████████████████████
best_epoch,4
best_precision,0.78986
improvement_vs_2k,32.88595
train/box_loss,1.34699



ÉXITO
Modelo: None
Revisa W&B dashboard para análisis completo
